# 🛒 실전 — 다양한 사이트 크롤링 연습

> **이 노트북에서 연습할 데이터 유형**
> 1. 🛍️ 쇼핑몰 상품 (로컬 HTML / books.toscrape.com)
> 2. 🔄 중고거래 (로컬 HTML 연습 후 → Selenium으로 실제 당근마켓)
> 3. 💬 텍스트/리뷰 데이터 (quotes.toscrape.com)
> 4. 💼 채용공고 (saramin.co.kr)
> 5. 📊 수집 데이터 합치기 & 저장

> ⚠️ 로컬 HTML 연습용 파일이 `data/` 폴더에 있습니다.
> VS Code에서 폴더를 열 때 `크롤링/` 폴더 자체를 열어야 상대경로가 맞아요!


---
## Part 1. 🛍️ 쇼핑몰 — 로컬 HTML 파일로 연습

**왜 로컬 파일로?**
- 실제 쇼핑몰(올리브영, 무신사 등)은 봇 차단/JS 렌더링으로 막힘
- 같은 HTML 구조를 `data/mock_shopping.html`로 미리 저장해뒀어요
- 네트워크 없이도 연습 가능!

**HTML 구조 미리보기:**
```html
<ol class="product-list">
  <li class="product-card" data-rank="1" data-product-id="P001">
    <div class="rank-badge">1</div>
    <div class="brand">라네즈</div>
    <h3 class="product-name">네오 쿠션 파운데이션</h3>
    <div class="price-area">
      <span class="original-price">38,000원</span>
      <span class="sale-price">27,000원</span>
      <span class="discount-rate">29%</span>
    </div>
    <div class="rating-area">
      <span class="stars">★★★★☆</span>
      <span class="review-count">(리뷰 1,204개)</span>
    </div>
    <a href="/products/P001" class="buy-link">구매하기</a>
  </li>
  ...
</ol>
```


In [3]:
from bs4 import BeautifulSoup
import pandas as pd
import re
import os

# 로컬 HTML 파일 읽기
# (VS Code에서 '크롤링' 폴더를 열었을 때의 상대경로)
html_path = os.path.join('data', 'mock_shopping.html')

with open(html_path, encoding='utf-8') as f:
    html_content = f.read()

soup = BeautifulSoup(html_content, 'html.parser')
print(f"파일 로드 완료! HTML 크기: {len(html_content):,} 글자")
print(f"페이지 제목: {soup.find('title').text}")


파일 로드 완료! HTML 크기: 3,946 글자
페이지 제목: 뷰티마켓 - 인기 화장품


In [4]:
# 쇼핑몰 데이터 추출

cards = soup.select('li.product-card')
print(f"상품 수: {len(cards)}개")

rows = []
for card in cards:
    # data-* 속성으로 순위와 상품ID 가져오기
    rank = card.get('data-rank')
    pid  = card.get('data-product-id')

    # 브랜드, 상품명
    brand = card.select_one('.brand')
    brand = brand.text.strip() if brand else None

    name  = card.select_one('.product-name')
    name  = name.text.strip() if name else None

    # 가격 (여러 span이 있으니 정확한 클래스로 골라야 함)
    original = card.select_one('.original-price')
    sale     = card.select_one('.sale-price')
    discount = card.select_one('.discount-rate')

    original = original.text.strip() if original else None
    sale     = sale.text.strip() if sale else None
    discount = discount.text.strip() if discount else '0%'

    # 리뷰수 — '(리뷰 1,204개)' 에서 숫자만 추출
    review = card.select_one('.review-count')
    if review:
        review_num = int(re.sub(r'[^0-9]', '', review.text) or 0)
    else:
        review_num = 0

    # 구매 링크
    link = card.select_one('.buy-link')
    href = link.get('href') if link else None

    rows.append({
        '순위': int(rank) if rank else None,
        '상품ID': pid,
        '브랜드': brand,
        '상품명': name,
        '정가': original,
        '할인가': sale,
        '할인율': discount,
        '리뷰수': review_num,
        '링크': href,
    })

df_shop = pd.DataFrame(rows)
print(df_shop.to_string(index=False))


상품 수: 5개
 순위 상품ID   브랜드                   상품명       정가     할인가 할인율  리뷰수             링크
  1 P001   라네즈  네오 쿠션 파운데이션 21호 아이보리  38,000원 27,000원 29% 1204 /products/P001
  2 P002 이니스프리   수분크림 그린티 씨드 세럼 80ml  25,000원 19,900원 20% 3891 /products/P002
  3 P003   설화수  자음 생크림 50ml 리미티드 에디션 120,000원 96,000원 20%  572 /products/P003
  4 P004  에스쁘아 프로 테일러 비이 파운데이션 SPF34  42,000원 42,000원  0%  887 /products/P004
  5 P005   클리오 킬 커버 더 뉴 파운웨어 쿠션 기획세트  32,000원 25,600원 20% 5023 /products/P005


In [ ]:
# 가격 문자열 → 숫자로 변환해서 분석

def price_to_int(price_str):
    """'27,000원' → 27000"""
    if not price_str:
        return None
    nums = re.sub(r'[^0-9]', '', price_str)
    return int(nums) if nums else None

df_shop['할인가_숫자'] = df_shop['할인가'].apply(price_to_int)
df_shop['정가_숫자']  = df_shop['정가'].apply(price_to_int)
df_shop['할인금액']   = df_shop['정가_숫자'] - df_shop['할인가_숫자']

print("=== 가격 분석 ===")
print(f"평균 할인가: {df_shop['할인가_숫자'].mean():,.0f}원")
print(f"최고 할인금액: {df_shop['할인금액'].max():,.0f}원")
print()
print(df_shop[['브랜드','상품명','정가_숫자','할인가_숫자','할인금액','리뷰수']].to_string(index=False))


---
## Part 2. 🔄 중고거래 — 로컬 HTML로 당근마켓 스타일 연습

**실제 당근마켓 / 번개장터는?**
- JavaScript 렌더링 → BeautifulSoup으로 빈 결과 나옴
- Selenium이 필요 (다음 수업에서!)

**지금은:** 실제와 동일한 HTML 구조를 로컬 파일로 연습합니다.

**HTML 구조 미리보기:**
```html
<li class="article-item" data-id="1001" data-category="디지털">
  <a href="/articles/1001" class="article-link">
    <h3 class="title">아이폰 15 프로 256GB</h3>
    <p class="price">1,200,000원</p>
    <span class="location">서울 강남구</span>
    <span class="time-ago">10분 전</span>
    <span class="badge status-sale">판매중</span>
    <span class="like-count">23</span>
  </a>
</li>
```


In [ ]:
# 중고거래 HTML 로드

html_path2 = os.path.join('data', 'mock_used_market.html')
with open(html_path2, encoding='utf-8') as f:
    html_used = f.read()

soup2 = BeautifulSoup(html_used, 'html.parser')
print(f"페이지 제목: {soup2.find('title').text}")

articles = soup2.select('li.article-item')
print(f"매물 수: {len(articles)}개")


In [ ]:
# 중고거래 데이터 추출

rows_used = []
for article in articles:
    # data-* 속성
    article_id = article.get('data-id')
    category   = article.get('data-category')

    # 제목, 가격
    title_el = article.select_one('.title')
    price_el = article.select_one('.price')

    title = title_el.text.strip() if title_el else None
    price = price_el.text.strip() if price_el else None

    # 지역, 시간
    location_el = article.select_one('.location')
    time_el     = article.select_one('.time-ago')

    location = location_el.text.strip() if location_el else None
    time_ago  = time_el.text.strip()    if time_el     else None

    # 상태 (판매중/예약중/판매완료)
    badge = article.select_one('.badge[class*="status-"]')
    if badge:
        classes = badge.get('class', [])
        # status-sale, status-reserved, status-sold 중 하나
        status_class = [c for c in classes if c.startswith('status-')]
        status_map = {
            'status-sale':     '판매중',
            'status-reserved': '예약중',
            'status-sold':     '판매완료',
        }
        status = status_map.get(status_class[0] if status_class else '', '알수없음')
    else:
        status = None

    # 가격제안 가능 여부
    nego = bool(article.select_one('.badge.nego'))

    # 좋아요 수
    like_el = article.select_one('.like-count')
    like_cnt = int(like_el.text.strip()) if like_el else 0

    # 링크
    link_el = article.select_one('a.article-link')
    href    = link_el.get('href') if link_el else None

    rows_used.append({
        'ID': article_id, '카테고리': category, '제목': title,
        '가격': price, '지역': location, '등록시간': time_ago,
        '상태': status, '가격제안': nego, '좋아요': like_cnt, '링크': href,
    })

df_used = pd.DataFrame(rows_used)
print(df_used[['카테고리','제목','가격','지역','상태','좋아요']].to_string(index=False))


In [ ]:
# 중고거래 데이터 분석

print("=== 카테고리별 현황 ===")
print(df_used.groupby('카테고리').agg(
    매물수=('ID', 'count'),
    평균좋아요=('좋아요', 'mean')
).round(1).to_string())

print()
print("=== 판매중인 매물만 ===")
on_sale = df_used[df_used['상태'] == '판매중']
on_sale['가격_숫자'] = on_sale['가격'].apply(price_to_int)
print(on_sale[['카테고리','제목','가격','좋아요']].to_string(index=False))

print()
print(f"=== 가격제안 가능 매물: {df_used['가격제안'].sum()}개 ===")
print(df_used[df_used['가격제안']][['제목','가격']].to_string(index=False))


---
## Part 3. 💬 텍스트/리뷰 데이터 — quotes.toscrape.com

리뷰, 댓글, 명언 같은 **텍스트 중심 데이터** 수집 연습입니다.
이 사이트는 크롤링 연습용으로 만들어져서 항상 동작합니다!

**HTML 구조:**
```html
<div class="quote">
  <span class="text">"The world as we have created it..."</span>
  <small class="author">Albert Einstein</small>
  <div class="tags">
    <a class="tag">change</a>
    <a class="tag">deep-thoughts</a>
  </div>
</div>
```


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'
}

# 1페이지 수집
url = 'http://quotes.toscrape.com/'
response = requests.get(url, headers=HEADERS)
response.encoding = 'utf-8'
print(f"상태 코드: {response.status_code}")

soup3 = BeautifulSoup(response.text, 'html.parser')
quotes = soup3.select('div.quote')
print(f"명언 수: {len(quotes)}개")


In [ ]:
# 명언 데이터 추출

def extract_quotes(soup_obj):
    rows = []
    for q in soup_obj.select('div.quote'):
        # 명언 텍스트 (앞뒤 " " 기호 포함되어 있음)
        text_el  = q.select_one('span.text')
        text     = text_el.text.strip() if text_el else None

        # 저자
        author_el = q.select_one('small.author')
        author    = author_el.text.strip() if author_el else None

        # 저자 상세 링크
        author_link = q.select_one('a')
        author_href = author_link.get('href') if author_link else None

        # 태그들 (여러 개)
        tags = [t.text.strip() for t in q.select('a.tag')]
        tags_str = ', '.join(tags)

        rows.append({
            '명언': text,
            '저자': author,
            '저자링크': author_href,
            '태그': tags_str,
            '태그수': len(tags),
        })
    return rows

all_quotes = extract_quotes(soup3)
df_quotes = pd.DataFrame(all_quotes)
print(df_quotes[['저자','명언','태그']].to_string(index=False))


In [ ]:
# 여러 페이지 수집 (자동 다음 페이지 감지)

all_quotes = []
page = 1

while page <= 5:
    url = f'http://quotes.toscrape.com/page/{page}/'
    response = requests.get(url, headers=HEADERS, timeout=10)
    response.encoding = 'utf-8'

    if response.status_code != 200:
        break

    soup_p = BeautifulSoup(response.text, 'html.parser')
    page_quotes = extract_quotes(soup_p)

    if not page_quotes:
        print(f"  {page}페이지에 데이터 없음, 종료")
        break

    all_quotes.extend(page_quotes)
    print(f"  {page}페이지: {len(page_quotes)}개 (누적: {len(all_quotes)}개)")

    # 다음 페이지 버튼이 있는지 확인
    next_btn = soup_p.select_one('li.next a')
    if not next_btn:
        print("  마지막 페이지 도달!")
        break

    page += 1
    time.sleep(1)

df_quotes_all = pd.DataFrame(all_quotes)
print(f"\n총 {len(df_quotes_all)}개 수집!")


In [ ]:
# 텍스트 데이터 분석

print("=== 저자별 명언 수 ===")
print(df_quotes_all['저자'].value_counts().head(5).to_string())

print()
print("=== 태그 순위 (전체 태그 중) ===")
all_tags = []
for tags_str in df_quotes_all['태그'].dropna():
    all_tags.extend([t.strip() for t in tags_str.split(',')])

import collections
tag_counter = collections.Counter(all_tags)
for tag, count in tag_counter.most_common(10):
    print(f"  {tag:20}: {count}회")


---
## Part 4. 💼 채용공고 — 여러 키워드 + 분석

사람인에서 **여러 직무 키워드**를 한번에 수집하고 비교해봅니다.


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0'
}

def crawl_saramin_simple(keyword, max_page=2):
    results = []
    for page in range(1, max_page + 1):
        url = (f'https://www.saramin.co.kr/zf_user/search/recruit'
               f'?searchType=search&searchword={keyword}&recruitPage={page}')
        try:
            resp = requests.get(url, headers=HEADERS, timeout=15)
            resp.encoding = 'utf-8'
            soup = BeautifulSoup(resp.text, 'html.parser')
            jobs = soup.select('div.item_recruit')

            for job in jobs:
                co = job.select_one('strong.corp_name a')
                ti = job.select_one('h2.job_tit a')
                conds = [c.text.strip() for c in job.select('div.job_condition span')]

                results.append({
                    '키워드': keyword,
                    '회사명': co.text.strip() if co else None,
                    '공고제목': ti.text.strip() if ti else None,
                    '지역': conds[0] if len(conds) > 0 else None,
                    '경력': conds[1] if len(conds) > 1 else None,
                    '학력': conds[2] if len(conds) > 2 else None,
                })

            print(f"  [{keyword}] {page}p → {len(jobs)}건")
            time.sleep(1)

        except Exception as e:
            print(f"  [{keyword}] {page}p 오류: {e}")
            break

    return results


# 직무 키워드 4종
keywords = ['데이터분석', '파이썬개발', 'UX디자인', '디지털마케팅']
all_jobs = []

for kw in keywords:
    jobs = crawl_saramin_simple(kw, max_page=2)
    all_jobs.extend(jobs)
    time.sleep(2)

df_jobs = pd.DataFrame(all_jobs)
print(f"\n총 {len(df_jobs)}건 수집 완료!")


In [ ]:
# 직무별 비교 분석

print("=== 직무별 공고 수 ===")
print(df_jobs['키워드'].value_counts().to_string())

print()
print("=== 지역별 공고 수 (상위 10) ===")
print(df_jobs['지역'].value_counts().head(10).to_string())

print()
print("=== 직무별 경력 조건 분포 ===")
pivot = df_jobs.groupby(['키워드', '경력']).size().unstack(fill_value=0)
# 경력무관, 신입, 신입·경력만 보기
entry_cols = [c for c in pivot.columns if c in ['경력무관', '신입', '신입·경력']]
if entry_cols:
    print(pivot[entry_cols].to_string())


---
## Part 5. 📚 books.toscrape.com — 카테고리별 수집

실제 온라인 서점처럼, **카테고리 별로** 데이터를 나눠서 수집합니다.


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

HEADERS = {'User-Agent': 'Mozilla/5.0'}
BASE_URL = 'http://books.toscrape.com'

# 카테고리 목록 가져오기
response = requests.get(BASE_URL, headers=HEADERS)
response.encoding = 'utf-8'
soup_main = BeautifulSoup(response.text, 'html.parser')

# 왼쪽 사이드바의 카테고리 링크들
category_links = soup_main.select('ul.nav-list li ul li a')
categories = []
for link in category_links[:8]:  # 8개만 (시간 절약)
    name = link.text.strip()
    href = link.get('href')
    categories.append({'이름': name, 'href': href})

print(f"발견된 카테고리: {len(category_links)}개 (상위 8개만 수집)")
for c in categories:
    print(f"  - {c['이름']}: {c['href']}")


In [ ]:
# 카테고리별 수집

rating_map = {'One':1,'Two':2,'Three':3,'Four':4,'Five':5}
all_books = []

for cat in categories:
    cat_url = BASE_URL + '/' + cat['href']
    resp = requests.get(cat_url, headers=HEADERS)
    resp.encoding = 'utf-8'
    soup_cat = BeautifulSoup(resp.text, 'html.parser')

    books = soup_cat.select('article.product_pod')
    for book in books:
        title_el  = book.select_one('h3 a')
        price_el  = book.select_one('p.price_color')
        rating_el = book.select_one('p.star-rating')

        title  = title_el['title'] if title_el else None
        price  = price_el.text.strip() if price_el else None
        rating = rating_map.get(
            rating_el['class'][1] if rating_el else '', 0
        )

        all_books.append({
            '카테고리': cat['이름'],
            '제목': title,
            '가격': price,
            '평점': rating,
        })

    print(f"  [{cat['이름']}] {len(books)}권")
    time.sleep(0.8)

df_books = pd.DataFrame(all_books)
print(f"\n총 {len(df_books)}권 수집!")


In [ ]:
# 카테고리별 분석

df_books['가격_숫자'] = (df_books['가격']
    .str.replace('£','').str.strip()
    .pipe(pd.to_numeric, errors='coerce'))

print("=== 카테고리별 평균 가격 & 평균 평점 ===")
analysis = df_books.groupby('카테고리').agg(
    책수=('제목','count'),
    평균가격=('가격_숫자','mean'),
    평균평점=('평점','mean')
).round(2).sort_values('평균평점', ascending=False)
print(analysis.to_string())

print()
print("=== 카테고리별 5점 책 수 ===")
five_star = df_books[df_books['평점']==5].groupby('카테고리').size()
print(five_star.sort_values(ascending=False).to_string())


---
## Part 6. 💾 전체 데이터 합치기 & 저장

지금까지 수집한 데이터를 각각 CSV로 저장합니다.


In [ ]:
import os

save_dir = os.path.join('data', 'results')
os.makedirs(save_dir, exist_ok=True)

# 각 데이터 저장
files = {
    '쇼핑몰_베스트.csv': df_shop,
    '중고거래_매물.csv': df_used,
    '명언_수집.csv': df_quotes_all,
    '채용공고_다직무.csv': df_jobs,
    '도서_카테고리별.csv': df_books,
}

for fname, df in files.items():
    path = os.path.join(save_dir, fname)
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(f"  ✅ {fname}: {len(df)}행 저장")

print(f"\n모두 {save_dir} 에 저장됐습니다!")


---
## 🎯 연습문제

### 문제 1 — 중고거래 (기초)
`data/mock_used_market.html`에서
- '서울' 지역 매물만 필터링하세요
- 좋아요 순으로 정렬하세요

### 문제 2 — 쇼핑몰 (응용)
`data/mock_shopping.html`에서
- 할인율이 20% 이상인 상품만 수집하세요
- 리뷰수가 1000개 이상인 상품은 '인기상품' 컬럼을 True로 표시하세요

### 문제 3 — 텍스트 (응용)
`quotes.toscrape.com`에서 전체 페이지를 수집하여
- 'love' 태그가 달린 명언만 모아서 출력하세요
- 가장 많이 인용된 저자 Top 3를 찾으세요

### 문제 4 — 채용공고 (실전)
사람인에서 '머신러닝' 키워드로 2페이지를 수집하고
- 제목에 'Python' 또는 '파이썬'이 들어간 공고만 필터링하세요
- 경력 조건별로 몇 건인지 집계하세요

<details>
<summary>문제 1 정답 보기</summary>

```python
# 서울 지역 필터링 + 좋아요 정렬
seoul = df_used[df_used['지역'].str.startswith('서울', na=False)]
seoul_sorted = seoul.sort_values('좋아요', ascending=False)
print(seoul_sorted[['제목','가격','지역','좋아요']].to_string(index=False))
```
</details>

<details>
<summary>문제 2 정답 보기</summary>

```python
# 할인율 숫자 변환
df_shop['할인율_숫자'] = df_shop['할인율'].str.replace('%','').astype(int)

# 20% 이상 필터
result = df_shop[df_shop['할인율_숫자'] >= 20].copy()

# 인기상품 컬럼 추가
result['인기상품'] = result['리뷰수'] >= 1000
print(result[['브랜드','상품명','할인율','리뷰수','인기상품']].to_string(index=False))
```
</details>
